# RIKUY_DEEP: deteccion de defectos en piezas de fundicion

Flujo: ingesta y versionado del dataset (data-v1) -> CNN (cnn-v1) -> ResNet50 (resnet-v1) -> comparacion en test -> exportacion a TensorFlow.js.

Decisiones de diseno:

El dataset de Kaggle ya viene augmentado: no se aplica augmentation durante el entrenamiento.
data-v1 es el dataset tal como se descarga. El redimensionado ocurre en tf.data, no se duplica en disco.
Clase positiva (1) = defecto. El modelo devuelve P(defecto).
Validacion se usa para early stopping y umbral; test solo para el reporte final.

In [1]:
%pip install -q kagglehub dvc dvc-gdrive tensorflowjs

## LIBRERIAS

In [2]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from datetime import datetime, timezone
from pathlib import Path
from PIL import Image
from google.colab import drive
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report,
                             confusion_matrix, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam

## CONFIGURACIÓN KAGGLE Y GOOGLE DRIVE

In [3]:
drive.mount('/content/drive')

DATA_VERSION, CNN_VERSION, RESNET_VERSION = 'data-v1', 'cnn-v1', 'resnet-v1'

KAGGLE_DATASET = 'ravirajsinh45/real-life-industrial-dataset-of-casting-product'

IMAGE_EXTENSIONS = {'.jpeg', '.jpg', '.png'}

PROJECT_DIR = Path('/content/drive/MyDrive/RIKUY_DEEP')
DATA_DIR = PROJECT_DIR / 'data'
RAW_DATA = DATA_DIR / 'casting_data'
REMOTE_DIR = PROJECT_DIR / 'dvc_remote'
REGISTRY_DIR = PROJECT_DIR / 'model_registry'
EXPORT_DIR = PROJECT_DIR / 'exports'
SPLITS_CSV = PROJECT_DIR / 'splits.csv'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
REMOTE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA.mkdir(parents=True, exist_ok=True)
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_CSV.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

print(f'Proyecto: {PROJECT_DIR}')
print(f'Datos: {DATA_DIR}')
print(f'Remoto DVC: {REMOTE_DIR}')
print(f'Data cruda: {RAW_DATA}')
print(f'Registro modelos: {REGISTRY_DIR}')
print(f'Modelos exportados: {EXPORT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto: /content/drive/MyDrive/RIKUY_DEEP
Datos: /content/drive/MyDrive/RIKUY_DEEP/data
Remoto DVC: /content/drive/MyDrive/RIKUY_DEEP/dvc_remote
Data cruda: /content/drive/MyDrive/RIKUY_DEEP/data/casting_data
Registro modelos: /content/drive/MyDrive/RIKUY_DEEP/model_registry
Modelos exportados: /content/drive/MyDrive/RIKUY_DEEP/exports


## Descargando el dataset

In [4]:
if not (RAW_DATA / 'def_front').exists() or not (RAW_DATA / 'ok_front').exists():
    print("Descargando dataset desde Kaggle con kagglehub...")
    path = Path(kagglehub.dataset_download(KAGGLE_DATASET))

    # Localizar la carpeta que contiene directamente las clases
    candidates = [
        path / 'casting_512x512' / 'casting_512x512',
        path / 'casting_512x512',
        path
    ]
    source_dir = next((c for c in candidates if (c / 'def_front').exists() and (c / 'ok_front').exists()), None)

    if source_dir is None:
        raise FileNotFoundError(f"No se encontró la carpeta 'casting_512x512' con las clases en {path}")

    print(f"Copiando imágenes 512x512 desde {source_dir} hacia {RAW_DATA}...")
    shutil.copytree(source_dir, RAW_DATA, dirs_exist_ok=True)
    print("Copia finalizada.")
else:
    print(f"ℹEl dataset 512x512 ya existe en {RAW_DATA}")
# Verificación y conteo de imágenes (Criterio de aceptación HU04)
num_def = len(list((RAW_DATA / 'def_front').glob('*.*')))
num_ok = len(list((RAW_DATA / 'ok_front').glob('*.*')))
total = num_def + num_ok

ℹEl dataset 512x512 ya existe en /content/drive/MyDrive/RIKUY_DEEP/data/casting_data


## Versionando el dataset

In [5]:
!mkdir -p /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2/data

In [6]:
!rsync -ah --progress /content/drive/MyDrive/RIKUY_DEEP/ /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2/data/

sending incremental file list
./
data/
data/casting_data/
data/casting_data/def_front/
data/casting_data/def_front/cast_def_0_0.jpeg
         26.38K 100%    0.00kB/s    0:00:00 (xfr#1, to-chk=1299/1309)
data/casting_data/def_front/cast_def_0_100.jpeg
         29.37K 100%   14.00MB/s    0:00:00 (xfr#2, to-chk=1298/1309)
data/casting_data/def_front/cast_def_0_1015.jpeg
         27.48K 100%    8.74MB/s    0:00:00 (xfr#3, to-chk=1297/1309)
data/casting_data/def_front/cast_def_0_102.jpeg
         27.07K 100%    2.15MB/s    0:00:00 (xfr#4, to-chk=1296/1309)
data/casting_data/def_front/cast_def_0_1046.jpeg
         26.38K 100%  660.51kB/s    0:00:00 (xfr#5, to-chk=1295/1309)
data/casting_data/def_front/cast_def_0_1053.jpeg
         24.51K 100%  569.89kB/s    0:00:00 (xfr#6, to-chk=1294/1309)
data/casting_data/def_front/cast_def_0_1055.jpeg
         30.51K 100%  252.50kB/s    0:00:00 (xfr#7, to-chk=1293/1309)
data/casting_data/def_front/cast_def_0_1077.jpeg
         30.31K 100%  246.66kB/s    

In [7]:
%cd /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2

/content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2


In [8]:
ls -l

total 202
drwx------ 7 root root   4096 Oct 29 03:46 data/
-rw------- 1 root root   7048 Oct 29 16:44 LICENSE
-rw------- 1 root root 195546 Oct 29 17:37 RIKUY_DEEP_G2.ipynb


In [10]:
!dvc init -f

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


In [12]:
# 1. Versionar la data ingestada con DVC
!dvc add data

# 2. Configurar usuario de Git (requerido por Colab)
!git config user.email "GIT_EMAIL"
!git config user.name "GIT_NAME"

# 3. Registrar únicamente los punteros de DVC y la configuración en Git
# Esto evita subir la carpeta pesada 'data/' a tu repositorio de GitHub
!git add data.dvc .gitignore .dvc/
!git commit -m "feat(RIKDP-15): HU04 · Ingestar y versionar el dataset"

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
  0% 0.00/781 [00:00<?, ?file/s{'info': ''}]
  4% 33.0/781 [00:00<00:02, 329file/s{'info': ''}]
  8% 63.0/781 [00:00<00:02, 308file/s{'info': ''}]
 12% 91.0/781 [00:00<00:02, 293file/s{'info': ''}]
 16% 124/781 [00:00<00:02, 306file/s{'info': ''}] 
 19% 152/781 [00:00<00:02, 292file/s{'info': ''}]
 23% 182/781 [00:00<00:02, 291file/s{'info': ''}]
 27% 211/781 [00:01<00:02, 264file/s{'info': ''}]
 31% 241/781 [00:01<00:01, 274file/s{'info': ''}]
 35% 276/781 [00:01<00:01, 296file/s{'info': ''}]
 42% 331/781 [00:01<00:01, 370file/s{'info': ''}]
 50% 394/781 [00:01<00:00, 446file/s{'info': ''}]
 58% 456/781 [00:01<00:00, 496file/s{'info': ''}]
 66% 513/781 [00:01<00:00, 516file/s{'info': ''}]
 73% 570/781 [00:01<00:00, 532file/s{'info': ''}]
 80% 625/781 [00:01<00:00, 537file/s{'info': ''}]
 87% 680/781 [00:01<00:00, 540file/s{'info': ''}]
 94% 735/781 [00:02<00:00, 528file/s{'info':

In [13]:
!git push origin RIKDP-15-HU04-Ingestar-y-versionar-el-dataset

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (8/8), 761 bytes | 42.00 KiB/s, done.
Total 8 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/d-a-b-o/RIKUY_DEEP_G2.git
   8de9299..2b29107  RIKDP-15-HU04-Ingestar-y-versionar-el-dataset -> RIKDP-15-HU04-Ingestar-y-versionar-el-dataset


In [14]:
!git add RIKUY_DEEP_G2.ipynb

total 226
drwx------ 7 root root   4096 Oct 29 03:46 data/
-rw------- 1 root root    107 Oct 29 17:40 data.dvc
-rw------- 1 root root   7048 Oct 29 16:44 LICENSE
-rw------- 1 root root 219009 Oct 29 17:46 RIKUY_DEEP_G2.ipynb
